## Чтение JSON и установка формата для плавающих чисел

In [98]:
import pandas as pd
import numpy as np

df = pd.read_json('data/auto.json', orient='records')
pd.options.display.float_format = '{:.2f}'.format
df.head(5)

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus


## Создание случайной выборки и объединение с исходным датафреймом

In [99]:
s_size = 200
smpl = df.sample(n=s_size, random_state=21, replace=True)
smpl['Fines'] = np.random.choice(df['Fines'], size=s_size, replace=True)
smpl['Refund'] = np.random.choice(df['Refund'], size=s_size, replace=True)
concat_rows = pd.concat([df, smpl], ignore_index=True)
print(df.shape)
print(concat_rows.shape)

(725, 5)
(925, 5)


## Добавление столбца Year со случайными данными

In [100]:
np.random.seed(21)
year = pd.Series(np.random.randint(1980, 2020, size=len(concat_rows)), name='Year')
fines = pd.concat([concat_rows, year], axis=1)
fines.head(5)

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,2,2000.00,Ford,Focus,2015
4,92918M178RUS,1,5700.00,Ford,Focus,2014


## Создание датафрейма с фамилиями

In [101]:
surns = pd.read_json('data/surname.json')
surns.columns = surns.iloc[0]
surns = surns[1:].reset_index(drop=True)

surnames = surns['NAME'].str.replace(r'[^A-Za-z]', '', regex=True)
np.random.seed(21)

c_uniq = concat_rows['CarNumber'].unique()
selected_surnames = pd.Series(np.random.choice(surnames, size=len(c_uniq), replace=True))

owners = pd.DataFrame({'CarNumber': c_uniq, 'SURNAME': selected_surnames.values})
owners.head(5)

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ


## Добавление наблюдений в датафреймы штрафов и владельцев

In [102]:
more_f_obs = pd.DataFrame({
    'CarNumber': ['SEV175MU999RUS', 'LUB121MU999RUS', 'PET169PI001RUS', 'DED173MU949RUS', 'BAB064OR940RUS'],
    'Make': ['BMW', 'Porsche', 'Audi', 'Toyota', 'Lada'],
    'Model': ['I7', 'Cayenne', 'RS8', 'RAV4', 'Zhiguli'],
    'Fines': ['9000', '150000', '10', '5000', '100'],
    'Refund': ['1', '2', '1', '2', '1'],
    'Year': ['2024', '2025', '2025', '2015', '2001']
})
fines = pd.concat([fines, more_f_obs], ignore_index=True)

owners = owners.drop(owners.index[-20:])
addit_obs = pd.DataFrame({
    'CarNumber': ['MOM073NE976RUS', 'MIR171MG176RUS', 'NEI113AA942RUS'],
    'SURNAME': ['Neizvestnova', 'Mironyuk', 'Neizvestnov']
})
owners = pd.concat([owners, addit_obs], ignore_index=True)

## Объединение датафреймов различными методами

In [103]:
df_in = pd.merge(fines, owners, on='CarNumber', how='inner')
df_out = pd.merge(fines, owners, on='CarNumber', how='outer')
df_l = pd.merge(fines, owners, on='CarNumber', how='left')
df_r = pd.merge(fines, owners, on='CarNumber', how='right')

## Создание сводной таблицы и сохранение в CSV

In [104]:
piv_f = pd.pivot_table(fines, index=['Make', 'Model'], columns='Year', values='Fines', aggfunc='sum', margins=False)
piv_f

Year                   1980      1981      1982     1983      1984      1985  \
Make       Model                                                               
Audi       RS8          NaN       NaN       NaN      NaN       NaN       NaN   
BMW        I7           NaN       NaN       NaN      NaN       NaN       NaN   
Ford       Focus   51794.59 423089.17 169372.93 61600.00 101694.59 131183.76   
           Mondeo       NaN       NaN       NaN      NaN       NaN       NaN   
Lada       Zhiguli      NaN       NaN       NaN      NaN       NaN       NaN   
Porsche    Cayenne      NaN       NaN       NaN      NaN       NaN       NaN   
Skoda      Octavia 16300.00       NaN   6900.00 11594.59   7000.00  10294.59   
Toyota     Camry   20594.59   8594.59       NaN  7200.00       NaN       NaN   
           Corolla      NaN       NaN   2000.00      NaN       NaN       NaN   
           RAV4         NaN       NaN       NaN      NaN       NaN       NaN   
Volkswagen Golf    30900.00       NaN       NaN  8594.59    300.00  24000.00   
           Jetta        NaN       NaN       NaN      NaN       NaN       NaN   
           Passat       NaN   4600.00       NaN  3200.00  10000.00   5000.00   
           Touareg      NaN       NaN       NaN      NaN       NaN   5800.00   

Year                   1986     1987      1988     1989  ...      2014  \
Make       Model                                         ...             
Audi       RS8          NaN      NaN       NaN      NaN  ...       NaN   
BMW        I7           NaN      NaN       NaN      NaN  ...       NaN   
Ford       Focus   86094.59 91800.00 121394.59 64800.00  ... 157894.59   
           Mondeo       NaN      NaN       NaN  8600.00  ...       NaN   
Lada       Zhiguli      NaN      NaN       NaN      NaN  ...       NaN   
Porsche    Cayenne      NaN      NaN       NaN      NaN  ...       NaN   
Skoda      Octavia   600.00 36300.00       NaN 91400.00  ...  17489.17   
Toyota     Camry        NaN      NaN       NaN 22400.00  ...       NaN   
           Corolla  1000.00  8000.00       NaN  4000.00  ...       NaN   
           RAV4         NaN      NaN       NaN      NaN  ...       NaN   
Volkswagen Golf         NaN  9300.00       NaN  6300.00  ...       NaN   
           Jetta        NaN      NaN       NaN      NaN  ...       NaN   
           Passat  15000.00 12300.00       NaN      NaN  ...       NaN   
           Touareg      NaN      NaN       NaN      NaN  ...   1300.00   

Year                    2015      2016      2017      2018     2019 2001  \
Make       Model                                                           
Audi       RS8           NaN       NaN       NaN       NaN      NaN  NaN   
BMW        I7            NaN       NaN       NaN       NaN      NaN  NaN   
Ford       Focus   223194.59 126194.59 311600.00 333183.76 69400.00  NaN   
           Mondeo        NaN  46200.00       NaN       NaN      NaN  NaN   
Lada       Zhiguli       NaN       NaN       NaN       NaN      NaN  100   
Porsche    Cayenne       NaN       NaN       NaN       NaN      NaN  NaN   
Skoda      Octavia  46394.59    300.00       NaN 156200.00  9500.00  NaN   
Toyota     Camry         NaN   1500.00   8594.59  13000.00 18100.00  NaN   
           Corolla       NaN       NaN   9600.00       NaN 36000.00  NaN   
           RAV4          NaN       NaN       NaN       NaN      NaN  NaN   
Volkswagen Golf      2300.00       NaN       NaN       NaN      NaN  NaN   
           Jetta         NaN       NaN       NaN       NaN      NaN  NaN   
           Passat   21200.00   2100.00       NaN       NaN      NaN  NaN   
           Touareg    500.00       NaN       NaN       NaN      NaN  NaN   

Year                2015  2024    2025  
Make       Model                        
Audi       RS8       NaN   NaN      10  
BMW        I7        NaN  9000     NaN  
Ford       Focus     NaN   NaN     NaN  
           Mondeo    NaN   NaN     NaN  
Lada       Zhiguli   NaN   NaN     NaN  
Porsche    Cayenne   NaN   NaN

In [105]:
fines.to_csv('data/fines.csv', index=False)
owners.to_csv('data/owners.csv', index=False)